In [2]:
# import funcs
%run ./utilsMassProfile.ipynb

In [ ]:
def step_to_skycoord(step_df):
    gc = SkyCoord(
        x=step_df["X"].values * u.pc,
        y=step_df["Y"].values * u.pc,
        z=step_df["Z"].values * u.pc,
        v_x=step_df["Vx"].values * (u.km/u.s),
        v_y=step_df["Vy"].values * (u.km/u.s),
        v_z=step_df["Vz"].values * (u.km/u.s),
        frame=Galactocentric()
    )
    return gc

def step_to_stream_coords(step_df):
    icrs = step_to_skycoord(step_df).icrs

    x,y,z = shrinking_sphere(*step_df[['X', 'Y', 'Z', 'M']].T.to_numpy())
    centre_deg = SkyCoord(x=x * u.pc,
                          y=y * u.pc,
                          z=z * u.pc,
                         frame=Galactocentric()
                    ).icrs 

    # finding rotation between average ra dec velocity (orbital) and x axis 
    #- https://stackoverflow.com/questions/6247153/angle-from-2d-unit-vector
    pmra_mean = np.median(icrs.pm_ra_cosdec.to_value())
    pmdec_mean = np.median(icrs.pm_dec.to_value())
    
    theta = np.arctan2(pmdec_mean, pmra_mean) * u.rad
    theta = theta.to(u.deg)
    tp = icrs.transform_to(centre_deg.skyoffset_frame(rotation=-theta))
    
    step_df["phi1"] = tp.lon.deg
    step_df["phi2"] = tp.lat.deg
    return tp


    